In [1]:
import pandas as pd

In [3]:
def compare_anomaly_models(*model_dfs, id_column='TestKey'):
    df_combined = pd.DataFrame({id_column: model_dfs[0][0][id_column]})
    
    # Voeg resultaten van elk model toe
    number_of_anomalies = 0
    model_names = []
    for df, model_name in model_dfs:
        number_of_anomalies +=1 
        model_names.append(model_name)
        df_combined = df_combined.merge(
            df[[id_column, 'is_anomaly']],
            on=id_column,
            suffixes=('', f'_{model_name}')
        )
        df_combined = df_combined.rename(columns={'is_anomaly': f'{model_name}_anomaly'})
    
    # Tel het aantal anomaliedetecties per datapunt
    anomaly_columns = [col for col in df_combined.columns if col.endswith('_anomaly')]
    df_combined['anomaly_count'] = df_combined[anomaly_columns].sum(axis=1)

    df_combined['anomaly_score'] = df_combined['anomaly_count'] / number_of_anomalies 
    
    return df_combined

In [6]:
## Test 
# import csvs
df_test_same_ans = pd.read_csv('./csv/same_answers_test_checked.csv')
df_test_time_spent = pd.read_csv('./csv/time_spent_test_checked.csv')
df_dbscan = pd.read_csv('./csv/dbscan_anomalies.csv')
df_neighbours = pd.read_csv('./csv/neighbours.csv')
# df_suod = pd.read_csv('./csv/suod.csv')
df_copod = pd.read_csv('./csv/copod.csv')
df_ecod = pd.read_csv('./csv/ecod.csv')
df_i_forest = pd.read_csv('./csv/i_forest.csv')
df_lof = pd.read_csv('./csv/lof.csv')
df_abod = pd.read_csv('./csv/abod.csv')

In [7]:
# time spent + same answers 
df_test_anomalies = df_test_same_ans.merge(df_test_time_spent)
df_test_anomalies.head()

,TestKey,SameAnswerPercentage,TimeSpent,TooSlow,TooFast
0,1.0,46.666667,129,False,True
1,2.0,28.070175,5892,True,False
2,3.0,27.272727,2606,False,False
3,4.0,31.578947,8922,True,False
4,5.0,33.333333,5776,True,False


In [54]:
# + DBSCAN 
# df_dbscan['DBSCAN'] = 1
# df_dbscan = df_dbscan[['TestKey', 'DBSCAN']]

# df_test_anomalies = df_test_anomalies.merge(df_dbscan, how='outer')

# df_test_anomalies['DBSCAN'].fillna(0, inplace=True)
# df_test_anomalies['DBSCAN'] = df_test_anomalies['DBSCAN'].astype(int)

# df_test_anomalies.head()

In [55]:
# + suod 

In [56]:
# + neighbours 
# df_neighbours = df_neighbours[['TestKey', 'spieker']]
# df_neighbours.rename(columns={'spieker': 'Neighbours'}, inplace=True)

# df_test_anomalies = df_test_anomalies.merge(df_neighbours)
# df_test_anomalies.head()

In [8]:
df_neighbours.rename(columns={'spieker': 'is_anomaly'}, inplace=True)
df_neighbours = df_neighbours[['TestKey', 'is_anomaly']]
df_neighbours.head()

,TestKey,is_anomaly
0,1,0
1,2,0
2,3,0
3,4,0
4,5,0


In [9]:
df_dbscan['is_anomaly'] = 1
df_dbscan = df_dbscan[['TestKey', 'is_anomaly']]

df_test_anomalies = df_test_anomalies.merge(df_dbscan, how='outer')

df_test_anomalies['is_anomaly'].fillna(0, inplace=True)
df_test_anomalies['is_anomaly'] = df_test_anomalies['is_anomaly'].astype(int)

df_dbscan = df_test_anomalies[['TestKey', 'is_anomaly']]

C:\Users\monad\AppData\Local\Temp\ipykernel_6640\133425726.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_test_anomalies['is_anomaly'].fillna(0, inplace=True)


In [10]:
df_dbscan.head()

,TestKey,is_anomaly
0,1.0,1
1,2.0,0
2,3.0,0
3,4.0,0
4,5.0,0


In [11]:
df_neighbours.rename(columns={'spieker': 'is_anomaly'}, inplace=True)
df_neighbours = df_neighbours[['TestKey', 'is_anomaly']]
df_neighbours.head()

,TestKey,is_anomaly
0,1,0
1,2,0
2,3,0
3,4,0
4,5,0


### Anomaly Score

In [12]:
df_test_same_ans['is_anomaly'] = df_test_same_ans.SameAnswerPercentage >= 50
df_test_same_ans.is_anomaly = df_test_same_ans.is_anomaly.astype(int)
df_test_same_ans.head()

,TestKey,SameAnswerPercentage,is_anomaly
0,1.0,46.666667,0
1,2.0,28.070175,0
2,3.0,27.272727,0
3,4.0,31.578947,0
4,5.0,33.333333,0


In [13]:
df_test_time_spent['is_anomaly'] = df_test_time_spent.TooFast | df_test_time_spent.TooSlow
df_test_time_spent.is_anomaly = df_test_time_spent.is_anomaly.astype(int)
df_test_time_spent.head()

,TestKey,TimeSpent,TooSlow,TooFast,is_anomaly
0,1.0,129,False,True,1
1,2.0,5892,True,False,1
2,3.0,2606,False,False,0
3,4.0,8922,True,False,1
4,5.0,5776,True,False,1


In [14]:
df_compare = compare_anomaly_models(
    (df_copod, 'copod'), 
    (df_ecod, 'ecod'), 
    (df_i_forest, 'i_forest'),
    (df_lof, 'lof'),
    # (df_suod,'suod'),
    (df_test_same_ans, 'same_ans'),
    (df_test_time_spent, 'timespent'),
    (df_dbscan, 'dbscan'),
    (df_neighbours, 'neighbours'),
    id_column='TestKey'
)
df_compare.head()

,TestKey,copod_anomaly,ecod_anomaly,i_forest_anomaly,lof_anomaly,same_ans_anomaly,timespent_anomaly,dbscan_anomaly,neighbours_anomaly,anomaly_count,anomaly_score
0,1.0,0,0,0,0,0,1,1,0,2,0.250
1,2.0,0,0,0,0,0,1,0,0,1,0.125
2,3.0,0,0,0,0,0,0,0,0,0,0.000
3,4.0,0,0,0,0,0,1,0,0,1,0.125
4,5.0,0,0,0,0,0,1,0,0,1,0.125


In [15]:
df_compare.to_csv('./csv/compare.csv', index=False)

In [65]:
# df_test_anomalies = df_test_anomalies.merge(df_compare[['TestKey', 'anomaly_score']], on='TestKey')
# df_test_anomalies

In [16]:
df_test_anomalies = df_compare.merge(df_test_anomalies[['TestKey', 'TooSlow','TooFast','SameAnswerPercentage']], on='TestKey')
df_test_anomalies.rename(columns={'SameAnswerPercentage':'SameAnswer'}, inplace=True)
df_test_anomalies.drop(columns=['timespent_anomaly', 'same_ans_anomaly'], inplace=True)

In [17]:
df_test_anomalies.head()

,TestKey,copod_anomaly,ecod_anomaly,i_forest_anomaly,lof_anomaly,dbscan_anomaly,neighbours_anomaly,anomaly_count,anomaly_score,TooSlow,TooFast,SameAnswer
0,1.0,0,0,0,0,1,0,2,0.250,False,True,46.666667
1,2.0,0,0,0,0,0,0,1,0.125,True,False,28.070175
2,3.0,0,0,0,0,0,0,0,0.000,False,False,27.272727
3,4.0,0,0,0,0,0,0,1,0.125,True,False,31.578947
4,5.0,0,0,0,0,0,0,1,0.125,True,False,33.333333


In [18]:
df_test_anomalies.to_csv("../../decoded_data/FCA/DimTestAnomalies.csv", index=False) 

### store in DimTestAnomalies

In [69]:
# df_test_anomalies.set_index("TestKey", inplace=True)
# df_test_anomalies.to_csv("../../decoded_data/FCA/DimTestAnomalies.csv")
# df_test_anomalies.head()

## QUESTION 
# import csvs


In [70]:
df_question_same_ans = pd.read_csv('./csv/same_answers_question_checked.csv')
df_question_time_spent = pd.read_csv('./csv/time_spent_question_checked.csv')


# same answer + time spent 


In [71]:
df_question_anomalies = df_question_time_spent.merge(df_question_same_ans)



# store in DimQuestionAnomalies


In [72]:
df_question_anomalies.rename(columns={"FCAQuestionKey":"QuestionKey"}, inplace=True)

In [73]:
df_question_anomalies.set_index("QuestionKey", inplace=True)
df_question_anomalies.to_csv('../../decoded_data/FCA/DimQuestionAnomalies.csv')